# 05 - Multi-Qubit Systems and Entanglement

**Prerequisites:** Notebooks 01-04. Familiarity with multi-qubit gates, especially CNOT.

Entanglement is the defining quantum resource -- the phenomenon that separates
quantum computing from any classical model of computation. When two qubits are
entangled, measuring one instantly determines the outcome of the other,
regardless of the distance between them. This correlation is stronger than
anything achievable classically, and it underpins quantum teleportation, super-
dense coding, quantum error correction, and most quantum algorithms.

By the end of this notebook you will be able to:

1. **Describe** entanglement and explain how it differs from classical correlation.
2. **Implement** all four Bell states and the GHZ state using Goqu.
3. **Predict** measurement correlations in entangled states.
4. **Explain** why entanglement does not enable faster-than-light communication.

In this notebook we will:

1. Understand **tensor products** and how multi-qubit states are represented.
2. Construct all four **Bell states** -- the simplest maximally entangled states.
3. Build **GHZ states** that extend entanglement to three or more qubits.
4. Use the **density matrix** formalism to quantify purity.
5. Address the common misconception that entanglement enables faster-than-light communication.

In [1]:
import (
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/sim/densitymatrix"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## Multi-Qubit States and Tensor Products

A single qubit lives in a 2-dimensional Hilbert space. When we have two qubits,
the combined system lives in the **tensor product** of the two individual spaces,
giving a 4-dimensional space with basis states |00>, |01>, |10>, |11>.

A **product state** (also called a *separable* state) is one that can be written
as a tensor product of individual qubit states:

$$|\psi\rangle = |\psi_0\rangle \otimes |\psi_1\rangle$$

An **entangled state** is any state that *cannot* be factored this way.

Let's start by preparing a product state: apply H to qubit 0 while leaving
qubit 1 untouched. The result should be $(|0\rangle + |1\rangle)/\sqrt{2} \otimes |0\rangle$.
In our bitstring convention $|q_1\,q_0\rangle$ (qubit 0 is the rightmost bit),
this is $(|00\rangle + |01\rangle)/\sqrt{2}$.

In [2]:
%%
// Product state: H on qubit 0, nothing on qubit 1
cProduct, err := builder.New("product", 2).
	H(0).
	Build()
if err != nil {
	panic(err)
}

fmt.Println("Circuit:")
gonbui.DisplayHTML(draw.SVG(cProduct))

sim := statevector.New(2)
if err := sim.Evolve(cProduct); err != nil {
	panic(err)
}

sv := sim.StateVector()
fmt.Println("Statevector (product state):")
for i, amp := range sv {
	if real(amp)*real(amp)+imag(amp)*imag(amp) > 1e-10 {
		fmt.Printf("  |%02b> : %.4f\n", i, amp)
	}
}
fmt.Println("\nThis is a product state: (|0>+|1>)/sqrt(2) x |0> = (|00>+|01>)/sqrt(2)")
fmt.Println("Qubit 0 (rightmost bit) is in superposition; qubit 1 (leftmost bit) is |0>.")
fmt.Printf("Each non-zero amplitude = 1/sqrt(2) = %.4f\n", 1/math.Sqrt(2))

Circuit:
Statevector (product state):
  |00> : (0.7071+0.0000i)
  |01> : (0.7071+0.0000i)

This is a product state: (|0>+|1>)/sqrt(2) x |0> = (|00>+|01>)/sqrt(2)
Qubit 0 (rightmost bit) is in superposition; qubit 1 (leftmost bit) is |0>.
Each non-zero amplitude = 1/sqrt(2) = 0.7071


q0 
 
 q1 
 
 H

## Bell States -- The Simplest Entangled States

The four **Bell states** form a maximally entangled basis for two qubits.
They are created by combining a Hadamard gate with a CNOT gate:

| Bell State | Circuit | Result |
|:---:|:---|:---|
| $|\Phi^+\rangle$ | H(0) then CNOT(0,1) on \|00> | $(|00\rangle + |11\rangle)/\sqrt{2}$ |
| $|\Phi^-\rangle$ | X(0), H(0), CNOT(0,1) on \|00> | $(|00\rangle - |11\rangle)/\sqrt{2}$ |
| $|\Psi^+\rangle$ | H(0), CNOT(0,1), X(1) on \|00> | $(|01\rangle + |10\rangle)/\sqrt{2}$ |
| $|\Psi^-\rangle$ | H(0), CNOT(0,1), X(1), Z(0) on \|00> | $(|01\rangle - |10\rangle)/\sqrt{2}$ |

The key insight: after the CNOT, the two qubits are **correlated in every
measurement basis**, not just the computational basis. This is what
distinguishes entanglement from classical correlation.

In [3]:
%%
// Build all four Bell states
bellCircuits := make(map[string]*ir.Circuit)

// Phi+ = (|00> + |11>) / sqrt(2)
c, err := builder.New("Phi+", 2).H(0).CNOT(0, 1).Build()
if err != nil {
	panic(err)
}
bellCircuits["Phi+"] = c

// Phi- = (|00> - |11>) / sqrt(2)
c, err = builder.New("Phi-", 2).X(0).H(0).CNOT(0, 1).Build()
if err != nil {
	panic(err)
}
bellCircuits["Phi-"] = c

// Psi+ = (|01> + |10>) / sqrt(2)
c, err = builder.New("Psi+", 2).H(0).CNOT(0, 1).X(1).Build()
if err != nil {
	panic(err)
}
bellCircuits["Psi+"] = c

// Psi- = (|01> - |10>) / sqrt(2)
c, err = builder.New("Psi-", 2).H(0).CNOT(0, 1).X(1).Z(0).Build()
if err != nil {
	panic(err)
}
bellCircuits["Psi-"] = c

// Evolve and print each Bell state
bellNames := []string{"Phi+", "Phi-", "Psi+", "Psi-"}
for _, name := range bellNames {
	bc := bellCircuits[name]
	fmt.Printf("=== Bell State |%s> ===\n", name)
	gonbui.DisplayHTML(draw.SVG(bc))

	sim := statevector.New(2)
	if err := sim.Evolve(bc); err != nil {
		panic(err)
	}

	sv := sim.StateVector()
	for i, amp := range sv {
		if real(amp)*real(amp)+imag(amp)*imag(amp) > 1e-10 {
			fmt.Printf("  |%02b> : %+.4f\n", i, real(amp))
		}
	}
	fmt.Println()
}

=== Bell State |Phi+> ===
  |00> : +0.7071
  |11> : +0.7071

=== Bell State |Phi-> ===
  |00> : +0.7071
  |11> : -0.7071

=== Bell State |Psi+> ===
  |01> : +0.7071
  |10> : +0.7071

=== Bell State |Psi-> ===
  |01> : -0.7071
  |10> : +0.7071



q0 
 
 q1 
 
 H

q0 
 
 q1 
 
 X 
 H

q0 
 
 q1 
 
 H 
 
 
 
 X

q0 
 
 q1 
 
 H 
 
 
 
 X 
 Z

## Perfect Correlation in Measurements

Let's measure the Bell state $|\Phi^+\rangle$ many times. Because the qubits
are maximally entangled, we should see only two outcomes: **|00>** and **|11>**,
each with roughly 50% probability. The outcomes |01> and |10> should
never appear -- the qubits are perfectly correlated.

In [4]:
%%
// Measure Bell Phi+ state 1000 times
cBellMeasure, err := builder.New("Phi+ measure", 2).
	H(0).
	CNOT(0, 1).
	MeasureAll().
	Build()
if err != nil {
	panic(err)
}

sim := statevector.New(2)
counts, err := sim.Run(cBellMeasure, 1000)
if err != nil {
	panic(err)
}

fmt.Println("Measurement results (1000 shots):")
for outcome, count := range counts {
	fmt.Printf("  |%s> : %d (%.1f%%)\n", outcome, count, float64(count)/10.0)
}
fmt.Println("\nOnly |00> and |11> appear -- perfect correlation!")

Measurement results (1000 shots):
  |00> : 498 (49.8%)
  |11> : 502 (50.2%)

Only |00> and |11> appear -- perfect correlation!


In [5]:
%%
// Visualize the measurement histogram
cBellHist, _ := builder.New("Phi+", 2).
	H(0).CNOT(0, 1).MeasureAll().Build()

simHist := statevector.New(2)
countsHist, _ := simHist.Run(cBellHist, 1000)

svg := viz.Histogram(countsHist, viz.WithTitle("Bell State |Phi+> Measurements"))
gonbui.DisplayHTML(svg)

Bell State |Phi+> Measurements 
 
 
 
 0 
 
 100 
 
 200 
 
 300 
 
 400 
 
 500 
 
 600 
 
 00 
 
 11

## GHZ State: Entanglement Beyond Two Qubits

The **Greenberger-Horne-Zeilinger (GHZ) state** extends Bell-style entanglement
to three or more qubits:

$$|\text{GHZ}\rangle = \frac{|000\rangle + |111\rangle}{\sqrt{2}}$$

The construction is straightforward: apply H to qubit 0, then cascade CNOT
gates to entangle each subsequent qubit. Like the Bell state, measurement
produces only "all zeros" or "all ones" outcomes.

In [6]:
%%
// Build a 3-qubit GHZ state: H(0), CNOT(0,1), CNOT(0,2)
cGHZ, err := builder.New("GHZ-3", 3).
	H(0).
	CNOT(0, 1).
	CNOT(0, 2).
	MeasureAll().
	Build()
if err != nil {
	panic(err)
}

fmt.Println("3-qubit GHZ circuit:")
gonbui.DisplayHTML(draw.SVG(cGHZ))

simGHZ := statevector.New(3)
countsGHZ, err := simGHZ.Run(cGHZ, 1000)
if err != nil {
	panic(err)
}

fmt.Println("Measurement results (1000 shots):")
for outcome, count := range countsGHZ {
	fmt.Printf("  |%s> : %d\n", outcome, count)
}

svgGHZ := viz.Histogram(countsGHZ, viz.WithTitle("3-Qubit GHZ State Measurements"))
gonbui.DisplayHTML(svgGHZ)

3-qubit GHZ circuit:
Measurement results (1000 shots):
  |111> : 511
  |000> : 489


q0 
 
 q1 
 
 q2 
 
 H 
 
 
 
 
 
 
 M 
 M 
 M

3-Qubit GHZ State Measurements 
 
 
 
 0 
 
 100 
 
 200 
 
 300 
 
 400 
 
 500 
 
 600 
 
 000 
 
 111

## Density Matrix and Purity

The **density matrix** $\rho$ is a more general representation of quantum states
that can describe both pure and mixed states. For a pure state $|\psi\rangle$:

$$\rho = |\psi\rangle\langle\psi|$$

The **purity** of a state is defined as $\text{Tr}(\rho^2)$:

- **Pure states** have purity = 1 (this includes entangled states!)
- **Mixed states** have purity < 1
- A **maximally mixed** state of dimension $d$ has purity = $1/d$

An important subtlety: a Bell state like $|\Phi^+\rangle$ is a **pure** state
(purity = 1) even though it is maximally entangled. Entanglement and mixedness
are different concepts. The mixedness (impurity) only appears when you trace
out one subsystem to look at the reduced density matrix of a single qubit.

In [7]:
%%
// Density matrix for the Bell state Phi+
cBellDM, err := builder.New("Phi+", 2).
	H(0).
	CNOT(0, 1).
	Build()
if err != nil {
	panic(err)
}

dm := densitymatrix.New(2)
if err := dm.Evolve(cBellDM); err != nil {
	panic(err)
}

purity := dm.Purity()
fmt.Printf("Purity of Bell state |Phi+> : %.4f\n", purity)
fmt.Printf("Is pure state (purity ~ 1.0): %v\n", math.Abs(purity-1.0) < 1e-10)
fmt.Println("\nDensity matrix (non-zero elements):")

rho := dm.DensityMatrix()
dim := 4 // 2^2 qubits
for r := 0; r < dim; r++ {
	for c := 0; c < dim; c++ {
		v := rho[r*dim+c]
		if real(v)*real(v)+imag(v)*imag(v) > 1e-10 {
			fmt.Printf("  rho[|%02b>,|%02b>] = %+.4f\n", r, c, real(v))
		}
	}
}

Purity of Bell state |Phi+> : 1.0000
Is pure state (purity ~ 1.0): true

Density matrix (non-zero elements):
  rho[|00>,|00>] = +0.5000
  rho[|00>,|11>] = +0.5000
  rho[|11>,|00>] = +0.5000
  rho[|11>,|11>] = +0.5000


In [8]:
%%
// Visualize the density matrix with a state city plot
cBellCity, _ := builder.New("Phi+", 2).
	H(0).CNOT(0, 1).Build()

dmCity := densitymatrix.New(2)
dmCity.Evolve(cBellCity)

rhoCity := dmCity.DensityMatrix()
svgCity := viz.StateCity(rhoCity, 4, viz.WithTitle("Density Matrix of |Phi+>"))
gonbui.DisplayHTML(svgCity)

Density Matrix of |Phi+> 
 Re(ρ) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 |00⟩ 
 |00⟩ 
 |01⟩ 
 |01⟩ 
 |10⟩ 
 |10⟩ 
 |11⟩ 
 |11⟩ 
 Im(ρ) 
 
 
 
 
 
 
 
 
 
 
 |00⟩ 
 |00⟩ 
 |01⟩ 
 |01⟩ 
 |10⟩ 
 |10⟩ 
 |11⟩ 
 |11⟩

## Predict, Then Verify

**Question:** If we measure qubit 0 of the Bell state $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ and get the result **|0>**, what do we know about qubit 1?

**Pause and predict** before running the next cell.

*Hint: Look at which basis states appear in the Bell state and what they imply about the relationship between the two qubits.*

In [9]:
%%
// Verify: measuring qubit 0 of Phi+ determines qubit 1
cVerify, _ := builder.New("Phi+ verify", 2).
	H(0).CNOT(0, 1).MeasureAll().Build()

simVerify := statevector.New(2)
countsVerify, _ := simVerify.Run(cVerify, 10000)

// Check that qubits always agree
totalShots := 0
agreedShots := 0
for outcome, count := range countsVerify {
	totalShots += count
	// In the bitstring, both bits should be the same
	if outcome[0] == outcome[1] {
		agreedShots += count
	}
}

fmt.Printf("Total shots:    %d\n", totalShots)
fmt.Printf("Qubits agreed:  %d\n", agreedShots)
fmt.Printf("Agreement rate:  %.1f%%\n", float64(agreedShots)/float64(totalShots)*100)
fmt.Println("\nPrediction confirmed: measuring qubit 0 completely determines qubit 1.")

Total shots:    10000
Qubits agreed:  10000
Agreement rate:  100.0%

Prediction confirmed: measuring qubit 0 completely determines qubit 1.


## Misconception: Entanglement Enables Faster-Than-Light Communication

A common misconception is that entanglement could be used to send information
instantaneously across any distance. The reasoning goes: "If Alice measures her
qubit and it instantly determines Bob's qubit, can't Alice use that to send a
message to Bob?"

**The answer is no.** This is guaranteed by the **no-communication theorem**.
Here is why:

1. **Alice cannot choose her measurement outcome.** When she measures her half
   of a Bell pair, she gets |0> or |1> with equal probability. She cannot force
   a particular result.

2. **Bob's local statistics are unchanged.** Before he learns Alice's result,
   Bob's qubit looks like a maximally mixed state -- completely random.
   No matter what Alice does (or doesn't do) to her qubit, Bob sees 50/50
   outcomes. He cannot tell whether Alice has measured or not.

3. **Correlation requires classical communication.** The correlations only become
   visible when Alice and Bob **compare their results**, which requires a
   classical communication channel limited by the speed of light.

Entanglement is a resource for protocols like quantum teleportation and
superdense coding, but these protocols always require a classical side-channel.
Quantum mechanics is non-local in its correlations, but **no information
travels faster than light**.

## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Can you distinguish |00⟩+|11⟩ from |00⟩-|11⟩ using only Z-basis measurements? Why or why not?
2. What measurement correlations prove entanglement versus classical correlation?
3. True or false: An entangled state is always a mixed state. Why?

## Exercises

### Exercise 1: 3-Qubit GHZ State Verification

Create a 3-qubit GHZ state using H(0), CNOT(0,1), CNOT(0,2).
Verify with 2000 measurement shots that only |000> and |111> appear.

In [10]:
%%
// Exercise 1: Build and verify a 3-qubit GHZ state
// GHZ = (|000> + |111>) / sqrt(2)
//
// TODO: Apply H to qubit 0, then cascade CNOT gates to entangle qubits 1 and 2
// cEx1, _ := builder.New("GHZ-3-ex", 3).
// 	/* your gates here */
// 	MeasureAll().
// 	Build()
//
// fmt.Println("GHZ-3 circuit:")
// fmt.Println(draw.String(cEx1))
//
// simEx1 := statevector.New(3)
// countsEx1, _ := simEx1.Run(cEx1, 2000)
//
// fmt.Println("Results (2000 shots):")
// allValid := true
// for outcome, count := range countsEx1 {
// 	fmt.Printf("  |%s> : %d\n", outcome, count)
// 	if outcome != "000" && outcome != "111" {
// 		allValid = false
// 	}
// }
// fmt.Printf("\nOnly |000> and |111> observed: %v\n", allValid)
//
// gonbui.DisplayHTML(viz.Histogram(countsEx1, viz.WithTitle("Exercise 1: GHZ-3 Measurements")))
fmt.Println("Uncomment the code above and fill in the gates!")

Uncomment the code above and fill in the gates!


### Exercise 2: Compose Two Circuits Using `ir.Tensor`

The `ir.Tensor` function places two circuits on **disjoint qubit spaces**.
If circuit A has $n$ qubits and circuit B has $m$ qubits, the result has
$n + m$ qubits. B's qubit indices are shifted by $n$.

Build two single-qubit circuits (H on qubit 0, and X on qubit 0),
then tensor them together to create a 2-qubit circuit that applies H to
qubit 0 and X to qubit 1.

In [11]:
%%
// Exercise 2: Use ir.Tensor to compose two independent circuits
// ir.Tensor places circuit A on qubits 0..n-1 and circuit B on qubits n..n+m-1
//
// TODO: Build two single-qubit circuits (H and X), then tensor them together
// cA, _ := builder.New("H-circuit", 1). /* gate */ .Build()
// cB, _ := builder.New("X-circuit", 1). /* gate */ .Build()
//
// // Tensor product: A on qubit 0, B on qubit 1
// cTensor := ir.Tensor(cA, cB)
//
// fmt.Println("Tensored circuit (H on q0, X on q1):")
// fmt.Println(draw.String(cTensor))
//
// simTensor := statevector.New(2)
// simTensor.Evolve(cTensor)
//
// svTensor := simTensor.StateVector()
// fmt.Println("Statevector:")
// for i, amp := range svTensor {
// 	p := real(amp)*real(amp) + imag(amp)*imag(amp)
// 	if p > 1e-10 {
// 		fmt.Printf("  |%02b> : %.4f  (prob = %.4f)\n", i, amp, p)
// 	}
// }
// fmt.Println("\nExpected: (|0>+|1>)/sqrt(2) tensor |1> = (|01>+|11>)/sqrt(2)")
// fmt.Println("This is a product state -- each qubit is independent.")
fmt.Println("Uncomment the code above and fill in the gates!")

Uncomment the code above and fill in the gates!


## Key Takeaways

1. **Tensor products** describe multi-qubit systems. An $n$-qubit state lives
   in a $2^n$-dimensional Hilbert space.

2. **Entangled states** cannot be factored into a tensor product of individual
   qubit states. This is fundamentally different from classical correlation.

3. The **Bell states** are the four maximally entangled 2-qubit states.
   They are built with H + CNOT and form a complete orthonormal basis.

4. **GHZ states** extend entanglement to $n$ qubits: $\frac{|00\ldots0\rangle + |11\ldots1\rangle}{\sqrt{2}}$.

5. The **density matrix** formalism generalizes state descriptions. **Purity**
   $\text{Tr}(\rho^2)$ distinguishes pure states (= 1) from mixed states (< 1).
   Entangled states can be pure!

6. **Entanglement does not enable faster-than-light communication.** The
   no-communication theorem guarantees that local measurements on one half of
   an entangled pair reveal no information about what was done to the other half.